# Signal-processing practice — executable Python companion
This is original practice based on the public MathWorks topic outlines. **It is not the proprietary Academy exercise set and does not establish MATLAB execution.** It provides independently checked resampling, frequency analysis, filtering, time-frequency plots, alignment, missing-value masks and subject-wise normalisation on synthetic data.

Public topics: https://matlabacademy.mathworks.com/details/signal-processing-onramp/signalprocessing


In [ ]:
import json
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy import signal
from sklearn.preprocessing import StandardScaler
from wearable_utils import make_wearable_windows,fit_signal_scaler,transform_signals
results=Path('results');results.mkdir(exist_ok=True)
data=pd.read_csv('data/synthetic_signal.csv');t=data.time_seconds.to_numpy()
raw=data.raw.to_numpy();reference=data.clean_reference.to_numpy();fs=200.
assert np.all(np.diff(t)>0)
sos=signal.butter(6,20,fs=fs,output='sos');filtered=signal.sosfiltfilt(sos,raw)
causal=signal.sosfilt(sos,raw)
raw_rmse=float(np.sqrt(np.mean((raw-reference)**2)))
filtered_rmse=float(np.sqrt(np.mean((filtered-reference)**2)))
assert filtered_rmse<raw_rmse
plt.plot(t[:600],raw[:600],alpha=.5,label='raw');plt.plot(t[:600],filtered[:600],label='filtered')
plt.xlabel('Time (s)');plt.ylabel('Amplitude');plt.legend();plt.show()
f,p=signal.welch(filtered,fs,nperseg=400)
peak_hz=float(f[np.argmax(p)])
assert abs(peak_hz-5)<.6
plt.semilogy(f,p);plt.xlabel('Frequency (Hz)');plt.ylabel('PSD (amplitude²/Hz)');plt.show()


## Resampling and explicit aliasing checks


In [ ]:
converted=signal.resample_poly(filtered,2,5)
assert len(converted)==int(len(filtered)*80/200)
t80=np.arange(400)/80
aliased=np.sin(2*np.pi*70*t80)
f_alias,p_alias=signal.periodogram(aliased,fs=80,nfft=4096)
alias_hz=float(f_alias[np.argmax(p_alias)])
assert abs(alias_hz-10)<.1
# Anti-aliasing suppresses the above-Nyquist 70 Hz component during downsampling.
high=np.sin(2*np.pi*70*t)
proper=signal.resample_poly(high,2,5)
assert np.sqrt(np.mean(proper[20:-20]**2))<.05
print({'new_samples':len(converted),'new_rate_hz':80,'alias_frequency_hz':alias_hz})


## Time-frequency analysis and clock alignment


In [ ]:
sweep=signal.chirp(t,5,t[-1],65)
fig,axes=plt.subplots(2,1,figsize=(9,6))
for axis,nperseg in zip(axes,[128,32]):
    f,ts,power=signal.spectrogram(sweep,fs,nperseg=nperseg,noverlap=nperseg//2)
    axis.pcolormesh(ts,f,10*np.log10(power+1e-12),shading='auto');axis.set_ylim(0,90)
    axis.set_ylabel('Hz');axis.set_title(f'STFT window: {nperseg} samples')
axes[-1].set_xlabel('Time (s)');plt.tight_layout();plt.show()
# Non-periodic pulse for an unambiguous alignment test; no circular wrapping.
base=signal.windows.gaussian(500,20);delay=17
delayed=np.concatenate([np.zeros(delay),base[:-delay]])
correlation=signal.correlate(delayed,base,mode='full');lags=signal.correlation_lags(len(delayed),len(base))
estimated_delay=int(lags[np.argmax(correlation)]);assert estimated_delay==delay
print('Estimated clock/sample lag:',estimated_delay,'samples')


## Keep missingness evidence and split participants before scaling


In [ ]:
x,y,subjects=make_wearable_windows();train=subjects<6;test=subjects>=6
original_missing=np.isnan(x);missing_rate=original_missing.mean(axis=(1,2))
usable=missing_rate<=.05
scaler=fit_signal_scaler(x[train & usable]);normalised=transform_signals(scaler,x[usable])
assert np.isfinite(normalised).all()
assert set(subjects[train]).isdisjoint(set(subjects[test]))
audit=pd.DataFrame({'subject':subjects,'missing_fraction':missing_rate,'retained':usable})
audit.to_csv(results/'synthetic_quality_audit.csv',index=False)
report={'raw_rmse':raw_rmse,'filtered_rmse':filtered_rmse,'dominant_hz':peak_hz,
    'alias_hz':alias_hz,'estimated_delay_samples':estimated_delay,
    'retained_windows':int(usable.sum()),'dataset':'synthetic','matlab_executed':False}
(results/'python_signal_checks.json').write_text(json.dumps(report,indent=2))
print(report)


## Preprocessing note for the research project
Filtering parameters depend on modality and acquisition rate; the settings in these exercises are demonstrations. Synchronise timestamps, keep original missingness/quality masks, split participants before fitting imputation and normalisation, and prevent overlapping windows from crossing evaluation boundaries. Interpolation and zero-phase filtering are offline operations; a deployed real-time system needs a separate causal pipeline. These files make no claims about the actual Emognition or WESAD data, which were not attached.
